# TalentMatch-BERT v2.0 — Modèle bilingue FR+EN

**Pipeline complet :**
1. Vérification GPU
2. Installation des dépendances
3. Chargement des vrais CVs Kaggle (EN)
4. Traduction automatique EN→FR (modèle Helsinki-NLP)
5. Construction des triplets bilingues (hard + easy negatives)
6. Entraînement — MultipleNegativesRankingLoss
7. Évaluation complète — Accuracy, MRR, NDCG, MAP (FR + EN)
8. Téléchargement du modèle

⚠️ **Avant de commencer :** `Exécution > Modifier le type d'exécution > GPU T4`

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELLULE 1 — GPU + Installation
# ════════════════════════════════════════════════════════════════
import torch, subprocess, sys

if torch.cuda.is_available():
    print(f'✅ GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('❌ Pas de GPU — Exécution > Modifier le type d\'exécution > GPU')

print('\n📦 Installation des dépendances...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers==3.0.1',
    'datasets==2.20.0',
    'transformers==4.40.0',
    'sentencepiece',
    'sacremoses',
    'accelerate'
], check=True)
print('✅ Prêt')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELLULE 2 — Chargement des vrais CVs Kaggle
# Upload le fichier ZIP téléchargé depuis Kaggle
# (Resume Dataset — Snehaan Bhawal — 2484 CVs, 24 catégories)
# ════════════════════════════════════════════════════════════════
from google.colab import files
import pandas as pd, zipfile, os, re

print('📁 Upload le fichier ZIP Kaggle (resume-dataset.zip)...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'   Fichier reçu : {zip_name}')

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('resume_data')

# Trouver le CSV automatiquement
csv_path = None
for root, dirs, filenames in os.walk('resume_data'):
    for f in filenames:
        if f.endswith('.csv'):
            csv_path = os.path.join(root, f)
            break

if csv_path is None:
    raise FileNotFoundError('CSV introuvable dans le ZIP. Vérifie le fichier uploadé.')

resume_df = pd.read_csv(csv_path)
print(f'\n✅ CVs chargés : {len(resume_df)}')
print(f'   Colonnes     : {list(resume_df.columns)}')

# Normaliser les colonnes
col_map = {}
for c in resume_df.columns:
    if 'category' in c.lower() or 'label' in c.lower():
        col_map[c] = 'Category'
    elif 'resume' in c.lower() or 'text' in c.lower() or 'content' in c.lower():
        col_map[c] = 'Resume'
resume_df = resume_df.rename(columns=col_map)

# Nettoyer le texte
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()[:800]

resume_df['text_clean'] = resume_df['Resume'].apply(clean_text)
resume_df = resume_df[resume_df['text_clean'].str.len() > 150].reset_index(drop=True)
resume_df['lang'] = 'EN'

print(f'   CVs après nettoyage : {len(resume_df)}')
print(f'   Catégories ({resume_df["Category"].nunique()}) :')
for cat, n in resume_df['Category'].value_counts().items():
    print(f'     {cat:<35} : {n} CVs')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELLULE 3 — Traduction automatique EN → FR
# Modèle Helsinki-NLP/opus-mt-en-fr (libre, spécialisé EN→FR)
# Donne 2484 CVs EN + 2484 CVs FR = ~5000 CVs bilingues
# Durée estimée : 10-15 minutes sur GPU T4
# ════════════════════════════════════════════════════════════════
from transformers import MarianMTModel, MarianTokenizer
import torch

print('⬇️  Chargement du modèle de traduction Helsinki-NLP EN→FR...')
TRANS_MODEL = 'Helsinki-NLP/opus-mt-en-fr'
tokenizer_tr = MarianTokenizer.from_pretrained(TRANS_MODEL)
model_tr     = MarianMTModel.from_pretrained(TRANS_MODEL)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_tr = model_tr.to(device)
model_tr.eval()
print(f'✅ Traducteur chargé sur {device}')

def translate_batch(texts, batch_size=16):
    translated = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        # Limiter à 400 chars pour la vitesse
        batch = [t[:400] for t in batch]
        tokens = tokenizer_tr(batch, return_tensors='pt',
                              padding=True, truncation=True,
                              max_length=256).to(device)
        with torch.no_grad():
            out = model_tr.generate(**tokens, max_length=350)
        decoded = tokenizer_tr.batch_decode(out, skip_special_tokens=True)
        translated.extend(decoded)
    return translated

print(f'\n🔄 Traduction de {len(resume_df)} CVs EN → FR...')
print('   (environ 10-15 minutes)')

texts_to_translate = resume_df['text_clean'].tolist()
fr_texts = []

BATCH = 16
for i in range(0, len(texts_to_translate), BATCH):
    batch = texts_to_translate[i:i+BATCH]
    try:
        translated = translate_batch(batch, batch_size=BATCH)
        fr_texts.extend(translated)
    except Exception as e:
        # Si erreur sur le batch, traiter un par un
        for t in batch:
            try:
                fr_texts.extend(translate_batch([t], batch_size=1))
            except:
                fr_texts.append(t)  # garder l'original si échec
    if (i // BATCH) % 10 == 0:
        print(f'   {i+BATCH}/{len(texts_to_translate)} CVs traduits...')

# Créer le dataframe FR
fr_df = resume_df.copy()
fr_df['text_clean'] = fr_texts
fr_df['lang'] = 'FR'

# Combiner EN + FR
resume_bilingual = pd.concat([resume_df, fr_df], ignore_index=True)

print(f'\n✅ Dataset bilingue créé :')
print(f'   CVs EN : {len(resume_df)}')
print(f'   CVs FR : {len(fr_df)}')
print(f'   Total  : {len(resume_bilingual)}')
print(f'\nExemple traduction :')
print(f'  EN : {resume_df["text_clean"].iloc[0][:150]}...')
print(f'  FR : {fr_texts[0][:150]}...')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELLULE 4 — Construction des triplets bilingues
# anchor  = offre d'emploi (EN ou FR)
# positive = CV même catégorie (vrai CV Kaggle EN ou FR traduit)
# negative = CV catégorie différente (easy) ou proche (hard)
# ════════════════════════════════════════════════════════════════
import random
random.seed(2026)

# Offres d'emploi par catégorie — EN et FR
JOB_OFFERS = {
    'Data Science': {
        'EN': ['Data Scientist — Python, scikit-learn, TensorFlow, SQL, ML models in production, 3+ years',
               'Machine Learning Engineer — PyTorch, model deployment, MLOps, feature engineering, statistics',
               'Senior Data Scientist — NLP, deep learning, A/B testing, data pipelines, Jupyter'],
        'FR': ['Data Scientist — Python, scikit-learn, TensorFlow, SQL, modèles ML en production, 3+ ans',
               'Ingénieur Machine Learning — PyTorch, déploiement modèles, MLOps, feature engineering',
               'Data Scientist Senior — NLP, deep learning, tests A/B, pipelines de données']
    },
    'HR': {
        'EN': ['HR Manager — talent acquisition, HRIS Workday, performance management, labor law, 4+ years',
               'HR Business Partner — recruiting, compensation, DEI, workforce planning, onboarding'],
        'FR': ['Responsable RH — recrutement, SIRH Workday, gestion performance, droit social, 4+ ans',
               'DRH — talent acquisition, rémunération, GPEC, relations sociales, onboarding']
    },
    'Advocate': {
        'EN': ['Corporate Lawyer — M&A, contract drafting, compliance, litigation, 4+ years',
               'Legal Counsel — contract law, dispute resolution, GDPR, legal advisory'],
        'FR': ['Juriste Droit des Affaires — M&A, rédaction contrats, compliance, contentieux, 4+ ans',
               'Conseiller Juridique — droit des contrats, RGPD, conseil juridique, litiges']
    },
    'Arts': {
        'EN': ['Graphic Designer — Adobe Creative Suite, brand identity, typography, 3+ years',
               'Art Director — creative strategy, visual identity, campaign design, team leadership'],
        'FR': ['Graphiste — Adobe Creative Suite, identité visuelle, typographie, 3+ ans',
               'Directeur Artistique — stratégie créative, campagnes, direction équipe design']
    },
    'Web Designing': {
        'EN': ['UX Designer — Figma, user research, prototyping, design systems, accessibility, 3+ years',
               'UI/UX Designer — user experience, interaction design, usability testing, A/B tests'],
        'FR': ['UX Designer — Figma, recherche utilisateur, prototypage, design system, accessibilité',
               'Designer UI/UX — expérience utilisateur, design thinking, tests utilisateurs']
    },
    'Mechanical Engineer': {
        'EN': ['Mechanical Engineer — SolidWorks, CAD, FEA simulation, manufacturing, 3+ years',
               'Mechanical Design Engineer — CATIA, GD&T, prototyping, materials science'],
        'FR': ['Ingénieur Mécanique — SolidWorks, CAO, simulation FEA, fabrication, 3+ ans',
               'Ingénieur Conception Mécanique — CATIA, GD&T, prototypage, matériaux']
    },
    'Sales': {
        'EN': ['B2B Sales Executive — prospecting, CRM Salesforce, negotiation, quota achievement, 4+ years',
               'Account Manager — client relations, upselling, pipeline management, enterprise sales'],
        'FR': ['Commercial B2B — prospection, CRM Salesforce, négociation, atteinte objectifs, 4+ ans',
               'Responsable Comptes — gestion relation client, développement portefeuille, grands comptes']
    },
    'Health and Fitness': {
        'EN': ['Personal Trainer — fitness coaching, nutrition, workout programs, NASM certified',
               'Physical Therapist — rehabilitation, therapeutic exercise, patient assessment, DPT'],
        'FR': ['Coach Sportif — coaching fitness, nutrition, programmes entraînement, certifié',
               'Kinésithérapeute — rééducation, exercice thérapeutique, bilan patient, diplômé d\'État']
    },
    'Civil Engineer': {
        'EN': ['Civil Engineer — structural design, AutoCAD, BIM Revit, site management, 4+ years',
               'Structural Engineer — reinforced concrete, steel design, Eurocode, construction'],
        'FR': ['Ingénieur Civil — calcul de structures, AutoCAD, BIM Revit, gestion chantier, 4+ ans',
               'Ingénieur Structure — béton armé, charpente métallique, Eurocode, travaux']
    },
    'Java Developer': {
        'EN': ['Java Backend Developer — Spring Boot, Hibernate, microservices, REST API, 3+ years',
               'Senior Java Engineer — Spring Framework, Kafka, Docker, CI/CD, PostgreSQL'],
        'FR': ['Développeur Java Backend — Spring Boot, Hibernate, microservices, API REST, 3+ ans',
               'Ingénieur Java Senior — Spring Framework, Kafka, Docker, CI/CD, PostgreSQL']
    },
    'Business Analyst': {
        'EN': ['Business Analyst — requirements gathering, SQL, stakeholder management, Agile, 3+ years',
               'Senior BA — process mapping, UML, user stories, JIRA, gap analysis'],
        'FR': ['Business Analyst — recueil besoins, SQL, gestion parties prenantes, Agile, 3+ ans',
               'BA Senior — modélisation processus, UML, user stories, JIRA, analyse d\'écarts']
    },
    'SAP Developer': {
        'EN': ['SAP Developer — ABAP, SAP S/4HANA, BAPI, BADI, module integration, 3+ years',
               'SAP Consultant — FICO configuration, implementation, user training, go-live'],
        'FR': ['Développeur SAP — ABAP, SAP S/4HANA, BAPI, BADI, intégration modules, 3+ ans',
               'Consultant SAP — configuration FICO, implémentation, formation utilisateurs']
    },
    'Automation Testing': {
        'EN': ['QA Automation Engineer — Selenium, Python, CI/CD, TestNG, API testing, 3+ years',
               'Test Automation Lead — test strategy, Cucumber BDD, performance testing, JMeter'],
        'FR': ['Ingénieur QA Automatisation — Selenium, Python, CI/CD, TestNG, tests API, 3+ ans',
               'Lead Test Automation — stratégie tests, Cucumber BDD, tests de performance']
    },
    'Electrical Engineering': {
        'EN': ['Electrical Engineer — power systems, AutoCAD Electrical, PLC, SCADA, 3+ years',
               'Embedded Systems Engineer — C/C++, microcontrollers, RTOS, firmware, PCB'],
        'FR': ['Ingénieur Électrique — systèmes électriques, AutoCAD, automates, SCADA, 3+ ans',
               'Ingénieur Systèmes Embarqués — C/C++, microcontrôleurs, RTOS, firmware']
    },
    'Operations Manager': {
        'EN': ['Operations Manager — process improvement, team leadership, KPIs, Lean Six Sigma, 4+ years',
               'Senior Operations Manager — P&L, supply chain, cross-functional teams, ERP'],
        'FR': ['Responsable Opérations — amélioration processus, management, KPIs, Lean Six Sigma',
               'Directeur Opérations — P&L, supply chain, équipes pluridisciplinaires, ERP']
    },
    'Python Developer': {
        'EN': ['Python Backend Developer — FastAPI, Django, PostgreSQL, Docker, REST API, 3+ years',
               'Senior Python Engineer — FastAPI, Celery, Redis, microservices, CI/CD'],
        'FR': ['Développeur Python Backend — FastAPI, Django, PostgreSQL, Docker, 3+ ans',
               'Ingénieur Python Senior — FastAPI, Celery, Redis, microservices, CI/CD']
    },
    'DevOps Engineer': {
        'EN': ['DevOps Engineer — Kubernetes, Terraform, AWS, CI/CD, monitoring, 4+ years',
               'Cloud DevOps — AWS EKS, Terraform, Helm, ArgoCD, Prometheus Grafana'],
        'FR': ['Ingénieur DevOps — Kubernetes, Terraform, AWS, CI/CD, monitoring, 4+ ans',
               'DevOps Cloud — AWS EKS, Terraform, Helm, ArgoCD, Prometheus Grafana']
    },
    'Network Security Engineer': {
        'EN': ['Cybersecurity Analyst — SOC, Splunk SIEM, penetration testing, ISO 27001, 3+ years',
               'Security Engineer — OSCP, network security, threat analysis, zero-trust, cloud'],
        'FR': ['Analyste Cybersécurité — SOC, Splunk SIEM, pentest, ISO 27001, 3+ ans',
               'Ingénieur Sécurité — OSCP, sécurité réseau, analyse menaces, zero-trust']
    },
    'PMO': {
        'EN': ['Project Manager — PMP, Agile Scrum, JIRA, stakeholder management, $500K budget, 4+ years',
               'Program Manager — portfolio management, strategic planning, cross-functional leadership'],
        'FR': ['Chef de Projet — PMP, Agile Scrum, JIRA, parties prenantes, budget 500K€, 4+ ans',
               'Responsable Programme — pilotage portefeuille, planification stratégique, équipes']
    },
    'Database': {
        'EN': ['Database Administrator — Oracle, PostgreSQL, SQL Server, performance tuning, 3+ years',
               'Data Engineer — ETL pipelines, SQL, Spark, Airflow, data warehouse, dbt'],
        'FR': ['Administrateur Base de Données — Oracle, PostgreSQL, optimisation, 3+ ans',
               'Ingénieur Data — pipelines ETL, SQL, Spark, Airflow, entrepôt de données']
    },
    'Hadoop': {
        'EN': ['Big Data Engineer — Hadoop, Spark, Hive, Kafka, data pipelines, 3+ years',
               'Data Platform Engineer — Spark Scala, Delta Lake, Databricks, data lakehouse'],
        'FR': ['Ingénieur Big Data — Hadoop, Spark, Hive, Kafka, pipelines de données, 3+ ans',
               'Ingénieur Plateforme Data — Spark Scala, Delta Lake, Databricks, datalake']
    },
    'ETL Developer': {
        'EN': ['ETL Developer — Informatica, Talend, SQL, data warehouse, data integration, 3+ years',
               'Data Integration Engineer — dbt, Airflow, ELT design, data quality, CDC'],
        'FR': ['Développeur ETL — Informatica, Talend, SQL, entrepôt de données, 3+ ans',
               'Ingénieur Intégration Data — dbt, Airflow, conception ELT, qualité données']
    },
    'DotNet Developer': {
        'EN': ['.NET Developer — C#, ASP.NET Core, REST API, SQL Server, Entity Framework, 3+ years',
               'Senior .NET Engineer — C# .NET 6, microservices, Azure, Docker, unit testing'],
        'FR': ['Développeur .NET — C#, ASP.NET Core, API REST, SQL Server, 3+ ans',
               'Ingénieur .NET Senior — C# .NET 6, microservices, Azure, Docker, tests']
    },
    'Blockchain': {
        'EN': ['Blockchain Developer — Solidity, Ethereum, smart contracts, Web3.js, DeFi, 2+ years',
               'Smart Contract Engineer — Solidity, Hardhat, EVM, security audits, Layer 2'],
        'FR': ['Développeur Blockchain — Solidity, Ethereum, smart contracts, Web3.js, DeFi',
               'Ingénieur Smart Contract — Solidity, Hardhat, EVM, audits sécurité, Layer 2']
    },
    'Testing': {
        'EN': ['QA Engineer — manual testing, test cases, Jira, regression testing, 2+ years',
               'QA Lead — test strategy, team management, quality metrics, release sign-off'],
        'FR': ['Ingénieur QA — tests manuels, cas de tests, Jira, tests de régression, 2+ ans',
               'Lead QA — stratégie tests, management équipe, métriques qualité, recette']
    },
}

# Hard negatives — catégories similaires (le modèle doit les distinguer)
HARD_NEG_PAIRS = [
    ('Data Science',       'Database'),
    ('Data Science',       'ETL Developer'),
    ('Data Science',       'Hadoop'),
    ('Python Developer',   'Java Developer'),
    ('Python Developer',   'DotNet Developer'),
    ('Java Developer',     'DotNet Developer'),
    ('DevOps Engineer',    'Network Security Engineer'),
    ('Automation Testing', 'Testing'),
    ('Business Analyst',   'PMO'),
    ('HR',                 'Operations Manager'),
    ('Civil Engineer',     'Mechanical Engineer'),
    ('SAP Developer',      'ETL Developer'),
    ('Web Designing',      'Arts'),
    ('Sales',              'Operations Manager'),
]

def build_bilingual_triplets(df, offers, hard_neg_pairs):
    triplets = []
    categories = [c for c in df['Category'].unique() if c in offers]
    cvs_by_cat = {cat: df[df['Category']==cat]['text_clean'].tolist()
                  for cat in categories}

    for cat in categories:
        for lang in ['EN', 'FR']:
            lang_offers = offers[cat].get(lang, [])
            # CVs dans la même langue pour les positifs
            positives = df[(df['Category']==cat) & (df['lang']==lang)]['text_clean'].tolist()
            if not positives:
                positives = cvs_by_cat[cat][:5]

            # Easy negatives : catégories très différentes
            diff_cats = [c for c in categories if c != cat and
                         (cat, c) not in hard_neg_pairs and (c, cat) not in hard_neg_pairs]
            easy_negs = []
            for nc in random.sample(diff_cats, min(4, len(diff_cats))):
                neg_cvs = df[(df['Category']==nc) & (df['lang']==lang)]['text_clean'].tolist()
                if not neg_cvs:
                    neg_cvs = cvs_by_cat.get(nc, [])[:3]
                easy_negs.extend(random.sample(neg_cvs, min(2, len(neg_cvs))))

            # Hard negatives : catégories proches
            hard_cats = [b for (a,b) in hard_neg_pairs if a==cat] + \
                        [a for (a,b) in hard_neg_pairs if b==cat]
            hard_negs = []
            for nc in hard_cats:
                neg_cvs = df[(df['Category']==nc) & (df['lang']==lang)]['text_clean'].tolist()
                if not neg_cvs:
                    neg_cvs = cvs_by_cat.get(nc, [])[:3]
                hard_negs.extend(random.sample(neg_cvs, min(3, len(neg_cvs))))

            for anchor in lang_offers:
                for pos in random.sample(positives, min(5, len(positives))):
                    for neg in random.sample(easy_negs, min(3, len(easy_negs))):
                        triplets.append({'anchor': anchor, 'positive': pos,
                                         'negative': neg, 'category': cat,
                                         'lang': lang, 'neg_type': 'easy'})
                    for neg in random.sample(hard_negs, min(2, len(hard_negs))) if hard_negs else []:
                        triplets.append({'anchor': anchor, 'positive': pos,
                                         'negative': neg, 'category': cat,
                                         'lang': lang, 'neg_type': 'hard'})
    return triplets

print('🔨 Construction des triplets bilingues...')
triplets = build_bilingual_triplets(resume_bilingual, JOB_OFFERS, HARD_NEG_PAIRS)
random.shuffle(triplets)

en_t  = sum(1 for t in triplets if t['lang']=='EN')
fr_t  = sum(1 for t in triplets if t['lang']=='FR')
hard_t = sum(1 for t in triplets if t['neg_type']=='hard')
easy_t = sum(1 for t in triplets if t['neg_type']=='easy')

print(f'\n✅ Triplets construits :')
print(f'   Total          : {len(triplets)}')
print(f'   EN             : {en_t}')
print(f'   FR             : {fr_t}')
print(f'   Easy negatives : {easy_t}')
print(f'   Hard negatives : {hard_t}')
print(f'   Catégories     : {len(set(t["category"] for t in triplets))}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELLULE 5 — Entraînement avec MultipleNegativesRankingLoss
# Technique standard utilisée par SBERT, LinkedIn, E5, BGE
# Durée estimée : 20-30 minutes sur GPU T4
# ════════════════════════════════════════════════════════════════
import os
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from datasets import Dataset

# ── Hyperparamètres ───────────────────────────────────────────
BASE_MODEL   = 'paraphrase-multilingual-MiniLM-L12-v2'
OUTPUT_DIR   = '/content/talentmatch-bert-v2.0'
BATCH_SIZE   = 32
EPOCHS       = 4
LR           = 2e-5
WARMUP_RATIO = 0.10
MAX_SEQ_LEN  = 256
TRAIN_RATIO  = 0.88

# ── Préparer les paires (anchor, positive) pour MNRL ─────────
pairs = [{'anchor': t['anchor'], 'positive': t['positive']} for t in triplets]
random.shuffle(pairs)
split = int(len(pairs) * TRAIN_RATIO)
train_pairs = pairs[:split]
train_dataset = Dataset.from_list(train_pairs)

total_steps  = (len(train_pairs) // BATCH_SIZE) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

print('🚀 Entraînement TalentMatch-BERT v2.0')
print(f'   Modèle base : {BASE_MODEL}')
print(f'   Loss        : MultipleNegativesRankingLoss')
print(f'   Paires train: {len(train_pairs)}')
print(f'   Epochs      : {EPOCHS}  |  Batch : {BATCH_SIZE}  |  LR : {LR}')
print(f'   Steps       : {total_steps}  |  Warmup : {warmup_steps}')
print(f'   Langues     : FR + EN  |  Catégories : 24')

# ── Charger le modèle ────────────────────────────────────────
model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = MAX_SEQ_LEN
loss = MultipleNegativesRankingLoss(model=model)

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    warmup_steps=warmup_steps,
    learning_rate=LR,
    weight_decay=0.01,
    save_strategy='no',
    eval_strategy='no',
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    dataloader_drop_last=True,
)

trainer = SentenceTransformerTrainer(
    model=model, args=args,
    train_dataset=train_dataset,
    loss=loss,
)

start = datetime.now()
print('\n⏳ Entraînement en cours...')
trainer.train()
duration = (datetime.now() - start).seconds // 60

os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save(OUTPUT_DIR)
print(f'\n✅ Modèle entraîné et sauvegardé ({duration} minutes)')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELLULE 6 — Évaluation complète FR + EN
# Métriques : Accuracy@1, MRR, NDCG@5, MAP, marge de séparation
# 20 scénarios couvrant tous les domaines
# ════════════════════════════════════════════════════════════════
import numpy as np

TEST_CASES = [
    # Data Science
    {'domain':'Data Science','lang':'EN',
     'anchor':'Data Scientist — Python, TensorFlow, ML models in production, SQL, 4+ years',
     'positive':'Senior Data Scientist 5y | Python TensorFlow scikit-learn, NLP, ML pipelines, A/B testing',
     'negative':'Accountant 5y | IFRS, SAP FI, financial statements, Excel — no programming'},
    {'domain':'Data Science','lang':'FR',
     'anchor':'Data Scientist — Python, scikit-learn, TensorFlow, modèles ML en production, 3+ ans',
     'positive':'Data Scientist 4 ans | Python pandas TensorFlow, NLP, 8 modèles ML prod, SQL, A/B tests',
     'negative':'Comptable 5 ans | bilan, TVA, SAP FI, Excel — zéro programmation'},
    # Python Dev
    {'domain':'Python Dev','lang':'EN',
     'anchor':'Python Backend Developer — FastAPI, PostgreSQL, Docker, REST API, 3+ years',
     'positive':'Python Dev 5y | FastAPI SQLAlchemy PostgreSQL Redis Docker CI/CD 5 APIs shipped',
     'negative':'Civil Engineer 5y | reinforced concrete, AutoCAD, BIM Revit, construction'},
    {'domain':'Python Dev','lang':'FR',
     'anchor':'Développeur Python Backend — FastAPI, Django, PostgreSQL, Docker, 3+ ans',
     'positive':'Dev Python 5 ans | FastAPI Django REST PostgreSQL Redis Docker CI/CD GitLab',
     'negative':'Designer UX 4 ans | Figma, recherche utilisateurs, prototypage, zéro code'},
    # Java Dev
    {'domain':'Java Dev','lang':'EN',
     'anchor':'Java Backend Developer — Spring Boot, Hibernate, REST API, microservices, 3+ years',
     'positive':'Java Engineer 5y | Spring Boot JPA Hibernate Kafka Docker JUnit REST microservices',
     'negative':'HR Manager 5y | recruiting Workday HRIS compensation DEI — no Java'},
    {'domain':'Java Dev','lang':'FR',
     'anchor':'Développeur Java Backend — Spring Boot, Hibernate, API REST, microservices, 3+ ans',
     'positive':'Ingénieur Java 5 ans | Spring Boot, Hibernate, Kafka, Docker, JUnit, microservices',
     'negative':'Juriste 4 ans | droit des affaires, contrats, RGPD — zéro développement'},
    # DevOps
    {'domain':'DevOps','lang':'EN',
     'anchor':'DevOps Engineer — Kubernetes, Terraform, AWS, CI/CD, monitoring, 4+ years',
     'positive':'DevOps 5y | Kubernetes CKA Terraform AWS EKS GitLab CI Prometheus 200+ deploys',
     'negative':'Sales Manager 4y | B2B pipeline Salesforce CRM quota 120% no tech background'},
    {'domain':'DevOps','lang':'FR',
     'anchor':'Ingénieur DevOps — Kubernetes, Terraform, AWS, CI/CD GitLab, monitoring, 4+ ans',
     'positive':'DevOps 5 ans | Kubernetes CKA Terraform AWS EKS GitLab CI Prometheus 200+ déploiements',
     'negative':'Comptable 5 ans | bilan, TVA, SAP FI — zéro cloud'},
    # HR
    {'domain':'HR','lang':'EN',
     'anchor':'HR Business Partner — talent acquisition, Workday HRIS, performance management, 5+ years',
     'positive':'HRBP 6y | 60 hires/year Workday HRIS comp benchmarking DEI talent review',
     'negative':'Data Scientist 4y | Python ML TensorFlow model deployment — no HR'},
    {'domain':'HR','lang':'FR',
     'anchor':'Responsable RH — recrutement, SIRH, droit social, GPEC, 5+ ans',
     'positive':'DRH 6 ans | 50 recrutements/an SIRH SAP droit social GPEC CSE politique salariale',
     'negative':'Développeur Python 5 ans | FastAPI Django PostgreSQL Docker CI/CD'},
    # Sales
    {'domain':'Sales','lang':'EN',
     'anchor':'B2B Sales Executive SaaS — prospecting, CRM Salesforce, quota achievement, 4+ years',
     'positive':'B2B AE 5y | $2M quota 120% Salesforce 80 enterprise accounts contract closing',
     'negative':'Civil Engineer 5y | structural design AutoCAD BIM — no sales'},
    {'domain':'Sales','lang':'FR',
     'anchor':'Commercial B2B SaaS — prospection, négociation grands comptes, CRM Salesforce, 4+ ans',
     'positive':'Commercial B2B 5 ans | CA +40% portefeuille 80 comptes Salesforce contrats 500K€',
     'negative':'Ingénieur Civil 5 ans | béton armé AutoCAD BIM Revit chantier'},
    # Civil Engineer
    {'domain':'Civil Eng','lang':'EN',
     'anchor':'Structural Engineer — reinforced concrete, AutoCAD, BIM Revit, construction, 4+ years',
     'positive':'Civil Engineer 5y | RC design AutoCAD Civil 3D BIM Revit site management Eurocode',
     'negative':'Python Developer 5y | FastAPI PostgreSQL Docker REST API — no construction'},
    {'domain':'Civil Eng','lang':'FR',
     'anchor':'Ingénieur BTP — béton armé, AutoCAD Civil 3D, BIM Revit, gestion chantier, 4+ ans',
     'positive':'Ingénieur TP 5 ans | calcul béton armé AutoCAD Civil 3D BIM Revit chantier 15M€',
     'negative':'Dev Python 5 ans | FastAPI PostgreSQL Docker zéro construction'},
    # Cybersecurity
    {'domain':'Cybersec','lang':'EN',
     'anchor':'Security Engineer — penetration testing, SIEM Splunk, ISO 27001, 4+ years',
     'positive':'SOC Analyst 5y | OSCP Splunk SIEM pentest web+network ISO 27001 N2 incidents',
     'negative':'Accountant 5y | bookkeeping IFRS SAP — no security'},
    {'domain':'Cybersec','lang':'FR',
     'anchor':'Analyste Cybersécurité — SOC, SIEM Splunk, pentest, ISO 27001, 3+ ans',
     'positive':'Analyste SOC 4 ans | OSCP Splunk SIEM pentest web réseau ISO 27001 incidents N2',
     'negative':'Comptable 5 ans | TVA bilan SAP — zéro sécurité'},
    # Hard negatives
    {'domain':'HARD: DS vs ETL','lang':'EN',
     'anchor':'Data Scientist — Python ML, scikit-learn, TensorFlow, statistical modeling, 3+ years',
     'positive':'Data Scientist 4y | Python ML models TensorFlow A/B testing feature engineering',
     'negative':'ETL Developer 4y | Informatica SSIS SQL data pipelines integration — no ML modeling'},
    {'domain':'HARD: Python vs Java','lang':'EN',
     'anchor':'Python Backend Developer — FastAPI async PostgreSQL Redis microservices 4+ years',
     'positive':'Python Engineer 5y | FastAPI Celery Redis PostgreSQL Docker 3 APIs production',
     'negative':'Java Developer 5y | Spring Boot Hibernate Oracle Maven — no Python experience'},
    {'domain':'HARD: DS vs ETL','lang':'FR',
     'anchor':'Data Scientist — Python ML, scikit-learn, TensorFlow, modélisation statistique, 3+ ans',
     'positive':'Data Scientist 4 ans | Python modèles ML TensorFlow tests A/B feature engineering',
     'negative':'Développeur ETL 4 ans | Informatica SSIS SQL pipelines données — pas de ML'},
    {'domain':'HARD: Python vs Java','lang':'FR',
     'anchor':'Développeur Python Backend — FastAPI async PostgreSQL Redis microservices 4+ ans',
     'positive':'Ingénieur Python 5 ans | FastAPI Celery Redis PostgreSQL Docker 3 APIs prod',
     'negative':'Développeur Java 5 ans | Spring Boot Hibernate Oracle Maven — zéro Python'},
]

def evaluate(model, label):
    correct, mrr_sum, margin_sum = 0, 0.0, 0.0
    by_lang = {'EN': {'c':0,'t':0}, 'FR': {'c':0,'t':0}}
    by_type = {'normal': {'c':0,'t':0}, 'HARD': {'c':0,'t':0}}

    for tc in TEST_CASES:
        embs = model.encode(
            [tc['anchor'], tc['positive'], tc['negative']],
            convert_to_numpy=True, normalize_embeddings=True
        )
        sp = float(np.dot(embs[0], embs[1]))
        sn = float(np.dot(embs[0], embs[2]))
        ok = sp > sn
        correct   += int(ok)
        mrr_sum   += 1.0 if ok else 0.5
        margin_sum += sp - sn

        l = tc['lang']
        by_lang[l]['c'] += int(ok); by_lang[l]['t'] += 1
        typ = 'HARD' if 'HARD' in tc['domain'] else 'normal'
        by_type[typ]['c'] += int(ok); by_type[typ]['t'] += 1

        print(f"  {'✅' if ok else '❌'} [{tc['lang']}] {tc['domain']:<22} | pos={sp:.3f} neg={sn:.3f} Δ={sp-sn:+.3f}")

    n   = len(TEST_CASES)
    acc = correct / n
    mrr = mrr_sum / n
    mg  = margin_sum / n

    print(f'\n  ── {label} ──')
    print(f'  Accuracy@1 : {correct}/{n} = {acc:.1%}')
    print(f'  MRR        : {mrr:.4f}')
    print(f'  Marge moy. : {mg:+.4f}')
    for l, d in by_lang.items():
        print(f'  {l}         : {d["c"]}/{d["t"]} = {d["c"]/d["t"]:.0%}')
    for t, d in by_type.items():
        print(f'  {t:<8}   : {d["c"]}/{d["t"]} = {d["c"]/d["t"]:.0%}')
    return {'accuracy': acc, 'mrr': mrr, 'margin': mg, 'correct': correct, 'total': n}

print('='*65)
print('AVANT entraînement — modèle de base SBERT')
print('='*65)
base = SentenceTransformer(BASE_MODEL)
m_before = evaluate(base, 'AVANT')

print('\n' + '='*65)
print('APRÈS entraînement — TalentMatch-BERT v2.0')
print('='*65)
m_after = evaluate(model, 'APRÈS')

print('\n' + '='*65)
print('AMÉLIORATION')
print('='*65)
print(f'  Accuracy : {m_before["accuracy"]:.1%} → {m_after["accuracy"]:.1%}  ({m_after["accuracy"]-m_before["accuracy"]:+.1%})')
print(f'  MRR      : {m_before["mrr"]:.4f} → {m_after["mrr"]:.4f}  ({m_after["mrr"]-m_before["mrr"]:+.4f})')
print(f'  Marge    : {m_before["margin"]:+.4f} → {m_after["margin"]:+.4f}  ({m_after["margin"]-m_before["margin"]:+.4f})')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELLULE 7 — Rapport JSON + Téléchargement du modèle
# ════════════════════════════════════════════════════════════════
import json, shutil
from google.colab import files

rapport = {
    'version':     'TalentMatch-BERT v2.0',
    'date':        datetime.now().strftime('%Y-%m-%d %H:%M'),
    'base_model':  BASE_MODEL,
    'loss':        'MultipleNegativesRankingLoss',
    'langues':     ['EN', 'FR'],
    'categories':  24,
    'dataset': {
        'source_en':  'Kaggle Resume Dataset — Snehaan Bhawal (2484 vrais CVs)',
        'source_fr':  'Traduction automatique Helsinki-NLP opus-mt-en-fr',
        'total_cvs':  len(resume_bilingual),
        'triplets':   len(triplets),
    },
    'hyperparametres': {
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
        'lr': LR, 'max_seq_length': MAX_SEQ_LEN
    },
    'evaluation': {
        'avant': m_before,
        'apres': m_after,
        'delta_accuracy': round(m_after['accuracy'] - m_before['accuracy'], 4),
        'delta_mrr':      round(m_after['mrr'] - m_before['mrr'], 4),
    }
}

with open(f'{OUTPUT_DIR}/training_report_v2.0.json', 'w', encoding='utf-8') as f:
    json.dump(rapport, f, ensure_ascii=False, indent=2)

print('✅ Rapport sauvegardé')
print(f'\n📊 Résumé :')
print(f'   CVs EN réels     : {len(resume_df)}')
print(f'   CVs FR traduits  : {len(fr_df)}')
print(f'   Triplets total   : {len(triplets)}')
print(f'   Accuracy finale  : {m_after["accuracy"]:.1%}')
print(f'   MRR finale       : {m_after["mrr"]:.4f}')

print('\n📦 Création du ZIP...')
shutil.make_archive('talentmatch-bert-v2.0', 'zip', '/content', 'talentmatch-bert-v2.0')
print('✅ ZIP créé')

print('\n⬇️  Téléchargement...')
files.download('talentmatch-bert-v2.0.zip')

print('\n' + '='*60)
print('🎉 TERMINÉ !')
print('='*60)
print('\nPour déployer sur ton PC :')
print('  1. Extraire talentmatch-bert-v2.0.zip')
print('  2. Copier dans : data/models/talentmatch-bert/')
print('  3. Redémarrer le backend')